# 11. Hybrid Search Mode

---

This notebook demonstrates **hybrid search mode**, which combines sparse (lexical) and
dense (semantic) retrieval for the highest concept mapping accuracy.

Portiere's hybrid backend fuses results from multiple retrieval backends using
Reciprocal Rank Fusion (RRF). You can now combine **any** supported dense backend
(FAISS, ChromaDB, PGVector, Qdrant, Milvus, MongoDB) with a sparse backend
(BM25s, Elasticsearch) via the `hybrid_backends` parameter.

**What you'll learn:**
- Configure hybrid search with `hybrid_backends`
- Combine different backend pairs (BM25s + FAISS, BM25s + ChromaDB, etc.)
- Tune fusion parameters (RRF k, weighted blending)
- Compare hybrid vs single-backend accuracy

**Prerequisites:**
- `pip install portiere[faiss]` or `pip install portiere[chromadb]` (depending on backend choice)

## 1. Setup

In [1]:
from pathlib import Path

# Sample data
ASSETS_DIR = Path("example_assets/11_hybrid_mode")
PATIENTS_CSV = ASSETS_DIR / "patients.csv"
DIAGNOSES_CSV = ASSETS_DIR / "diagnoses.csv"

for f in [PATIENTS_CSV, DIAGNOSES_CSV]:
    assert f.exists(), f"Missing: {f}"
print("All sample data files found.")

All sample data files found.


In [2]:
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        hybrid_backends=["bm25s", "faiss"],
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


## 2. Configure Hybrid Search with `hybrid_backends`

The `hybrid_backends` parameter explicitly specifies which sub-backends to combine.
This replaces the old auto-detection pattern where hybrid mode always used FAISS + BM25s.

In [3]:
import portiere
from portiere.config import (
    PortiereConfig,
    KnowledgeLayerConfig,
    EmbeddingConfig,
    RerankerConfig,
)
from portiere.engines import PolarsEngine

# Hybrid: BM25s (lexical) + FAISS (semantic) — the classic combination
config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        hybrid_backends=["bm25s", "faiss"],
        bm25s_corpus_path=str(BM25S_CORPUS),
        faiss_index_path=str(FAISS_INDEX),
        faiss_metadata_path=str(FAISS_META),
        fusion_method="rrf",
        rrf_k=60,
    ),
    embedding=EmbeddingConfig(
        provider="huggingface",
        model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
    ),
    reranker=RerankerConfig(provider="none"),
)

project = portiere.init(
    name="Hybrid Search Demo",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    config=config,
)

print(project)
print(f"Mode: {config.effective_mode}")
print(f"Backend: {config.knowledge_layer.backend}")
print(f"Sub-backends: {config.knowledge_layer.hybrid_backends}")

2026-04-16 00:25:38 [info     ] PolarsEngine initialized      


2026-04-16 00:25:38 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project(name='Hybrid Search Demo', task='standardize', target_model='omop_cdm_v5.4', mode='local')
Mode: local
Backend: hybrid
Sub-backends: ['bm25s', 'faiss']


## 3. Alternative Backend Combinations

You can combine any dense backend with any sparse backend. Here are some common pairings:

In [4]:
# Example: BM25s + ChromaDB
config_chroma = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        hybrid_backends=["bm25s", "chromadb"],
        bm25s_corpus_path=str(BM25S_CORPUS),
        chroma_persist_path="./data/vocab/chroma",
        fusion_method="rrf",
    ),
)
print(f"ChromaDB hybrid: {config_chroma.knowledge_layer.hybrid_backends}")

# Example: BM25s + Qdrant
config_qdrant = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        hybrid_backends=["bm25s", "qdrant"],
        bm25s_corpus_path=str(BM25S_CORPUS),
        qdrant_url="http://localhost:6333",
        fusion_method="rrf",
    ),
)
print(f"Qdrant hybrid: {config_qdrant.knowledge_layer.hybrid_backends}")

# Example: Elasticsearch + Milvus
config_milvus = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        hybrid_backends=["elasticsearch", "milvus"],
        elasticsearch_url="http://localhost:9200",
        milvus_uri="./data/vocab/milvus_portiere.db",
        fusion_method="rrf",
    ),
)
print(f"Milvus hybrid: {config_milvus.knowledge_layer.hybrid_backends}")

ChromaDB hybrid: ['bm25s', 'chromadb']
Qdrant hybrid: ['bm25s', 'qdrant']
Milvus hybrid: ['elasticsearch', 'milvus']


## 4. Run the Pipeline with Hybrid Search

All pipeline operations work the same regardless of backend. The hybrid backend is transparent to the rest of the pipeline.

In [5]:
# Add sources
patients = project.add_source(str(PATIENTS_CSV))
diagnoses = project.add_source(str(DIAGNOSES_CSV))

print(f"Added patients: {patients['row_count']} rows")
print(f"Added diagnoses: {diagnoses['row_count']} rows")

2026-04-16 00:25:38 [info     ] project.source_added           project='Hybrid Search Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:25:41 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:25:41 [info     ] project.profiled               source=patients


2026-04-16 00:25:41 [info     ] project.source_added           project='Hybrid Search Demo' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:25:41 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


2026-04-16 00:25:41 [info     ] project.profiled               source=diagnoses


Added patients: 15 rows
Added diagnoses: 20 rows


In [6]:
# Map schema
schema_map = project.map_schema(patients)
print("Schema mapping:")
print(schema_map.summary())

2026-04-16 00:25:41 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:25:41 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:25:41 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:25:41 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:25:43 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:25:43 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:25:43 [info     ] local_storage.schema_mapping_saved items_count=11 project='Hybrid Search Demo'


2026-04-16 00:25:43 [info     ] project.schema_mapped          auto=10 project='Hybrid Search Demo' total=11


Schema mapping:
{'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}


In [7]:
# Map concepts — uses hybrid search (BM25s + FAISS with RRF fusion)
concept_map = project.map_concepts(source=diagnoses)
print("Concept mapping:")
print(concept_map.summary())

2026-04-16 00:25:43 [info     ] project.code_columns_detected  columns=['diagnosis_code']


2026-04-16 00:25:43 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=hybrid


2026-04-16 00:25:43 [info     ] Mapping column: diagnosis_code


2026-04-16 00:25:43 [info     ] Found description column: diagnosis_description code_column=diagnosis_code desc_column=diagnosis_description mappings=8


2026-04-16 00:25:43 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


...

2026-04-16 00:25:56 [warning  ] hybrid.backend_failed          backend=LocalFAISSBackend error='sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers'


2026-04-16 00:25:56 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=8


2026-04-16 00:25:57 [info     ] local_storage.concept_mapping_saved items_count=8 project='Hybrid Search Demo'


2026-04-16 00:25:57 [info     ] project.concepts_mapped        auto_rate=100.0 project='Hybrid Search Demo' total=8


Concept mapping:
{'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}


## 5. Fusion Methods

Portiere supports two fusion strategies for combining results from multiple backends:

| Method | Parameter | Description |
|--------|-----------|-------------|
| `rrf` | `rrf_k` (default: 60) | Reciprocal Rank Fusion. Rank-based, parameter-free beyond `k`. |
| `weighted` | `fusion_weights` | Weighted linear combination of normalized scores. |

RRF is recommended as the default. It is robust and does not require tuning score distributions across backends.

In [8]:
# RRF fusion (default)
config_rrf = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        hybrid_backends=["bm25s", "faiss"],
        bm25s_corpus_path=str(BM25S_CORPUS),
        faiss_index_path=str(FAISS_INDEX),
        faiss_metadata_path=str(FAISS_META),
        fusion_method="rrf",
        rrf_k=60,
    ),
)
print(f"Fusion: {config_rrf.knowledge_layer.fusion_method}, k={config_rrf.knowledge_layer.rrf_k}")

# Weighted fusion
config_weighted = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        hybrid_backends=["bm25s", "faiss"],
        bm25s_corpus_path=str(BM25S_CORPUS),
        faiss_index_path=str(FAISS_INDEX),
        faiss_metadata_path=str(FAISS_META),
        fusion_method="weighted",
        fusion_weights=[0.4, 0.6],  # 40% BM25s, 60% FAISS
    ),
)
print(f"Fusion: {config_weighted.knowledge_layer.fusion_method}, weights={config_weighted.knowledge_layer.fusion_weights}")

Fusion: rrf, k=60
Fusion: weighted, weights=[0.4, 0.6]


## 6. Inspect Local Artifacts

All artifacts are stored locally as YAML/CSV files that you can inspect and edit.

In [9]:
import os

# Show the local project directory structure
project_dir = config.local_project_dir / "Hybrid Search Demo"
print(f"Project directory: {project_dir}")
print()

for root, dirs, files in os.walk(project_dir):
    level = root.replace(str(project_dir), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        filepath = Path(root) / file
        size = filepath.stat().st_size
        print(f"{sub_indent}{file} ({size:,} bytes)")

Project directory: /Users/kuangsmacbook/.portiere/projects/Hybrid Search Demo

Hybrid Search Demo/
  project.yaml (265 bytes)
  .DS_Store (8,196 bytes)
  cross_mappings/
  concept_mappings/
    source_to_concept_map.csv (12,657 bytes)
    concept_mapping.yaml (14,033 bytes)
  validation_reports/
  schema_mappings/
    schema_mapping.yaml (2,410 bytes)
  etl_scripts/
  quality_reports/
    location_20260219_095540.json (1,148 bytes)
    person_20260219_095540.json (6,465 bytes)
    location_20260218_190342.json (1,148 bytes)
    person_20260415_161655.json (7,012 bytes)
    person_20260218_190342.json (6,465 bytes)
    location_20260415_161655.json (6,204 bytes)
  sources/
    diagnoses.yaml (249 bytes)
    patients.yaml (262 bytes)
  profiles/
    diagnoses.json (7,714 bytes)
    patients.json (14,104 bytes)


## 7. Backend Pairing Guide

| Sparse Backend | Dense Backend | Use Case |
|:---|:---|:---|
| BM25s | FAISS | Default. Best local-only accuracy, no external services. |
| BM25s | ChromaDB | Persistent local vectors without managing FAISS files. |
| BM25s | Qdrant | High-throughput search with payload filtering. |
| BM25s | Milvus (Lite) | Local file-based vectors, scales to billions if needed. |
| Elasticsearch | FAISS | Production lexical search + semantic matching. |
| Elasticsearch | PGVector | All-PostgreSQL stack for teams with existing infra. |
| Elasticsearch | MongoDB | MongoDB-centric teams needing lexical + semantic. |

---

> **Portiere Cloud:** For team collaboration features (push/pull, dashboard review,
> real-time sync), see [Portiere Cloud](https://portiere.io). The open-source SDK
> focuses on local-only operation with full support for all vector store backends.

## Summary

Hybrid search combines lexical and semantic retrieval for the best mapping accuracy:

1. **Configure** -- set `backend="hybrid"` with `hybrid_backends` to pick your sub-backends
2. **Build indexes** -- `build_knowledge_layer()` creates indexes for both backends
3. **Map** -- the pipeline automatically fuses results from both backends via RRF
4. **Tune** -- adjust `rrf_k` or switch to `weighted` fusion if needed

**Next steps:**
- See `01.3_hybrid_search_reranking.ipynb` for hybrid search + cross-encoder reranking
- See `04_knowledge_backends.ipynb` for detailed configuration of all 9 backends
- See `12_mapping_review_workflow.ipynb` for approve/reject/override workflow